# hopmil — runner Kaggle (E2: experimentos de imagem em GPU)

Roda a comparação dos 4 agregadores nos datasets de imagem, na GPU do Kaggle,
logando no W&B (projeto `artigo-3`) e salvando os CSVs em `/kaggle/working` para
download. Se houver **2 GPUs** (T4 x2), roda **um dataset por GPU em paralelo**.

## Antes de rodar (uma vez)
1. **Settings → Accelerator → GPU T4 x2** (2 GPUs → mnist roda numa, cifar noutra,
   em paralelo). Com 1 GPU também funciona (sequencial).
2. **Settings → Internet → On** (para clonar o repo e baixar MNIST/CIFAR).
3. **Add-ons → Secrets**: criar `WANDB_API_KEY` com sua chave do W&B.
4. **Add Input**: adicionar os datasets reais de histopatologia (colon / UCSB)
   como *Datasets* do Kaggle e ajustar `COLON_ROOT` / `UCSB_ROOT` abaixo.

## Fluxo
- Rode com `QUICK = True` primeiro: faz um teste minúsculo em **todos** os
  datasets, no projeto `artigo-3-smoke`, só para confirmar que loga certo.
- Depois `QUICK = False`: roda o protocolo completo (**5×10-fold**, `NREPEATS=5`)
  no projeto `artigo-3`.
- Cada job escreve um log próprio em `/kaggle/working/log_<dataset>.txt`; a célula
  imprime um *heartbeat* (última linha) de cada GPU a cada 20s.

In [ ]:
# --- Parâmetros (edite aqui) ---
REPO_URL = "https://github.com/waks38/artigo3.git"
BRANCH = "master"

# Raízes dos datasets reais (mounts de /kaggle/input). Se não existirem, são puladas.
COLON_ROOT = "/kaggle/input/crchistophenotypes/CRCHistoPhenotypes_2016_04_28"
# UCSB: o dataset andrewmvd monta aninhado em datasets/<autor>/<slug> e as imagens
# .tif ficam em Images/ (as .TIF maiúsculas em Masks/ são ignoradas pelo loader).
UCSB_ROOT = "/kaggle/input/datasets/andrewmvd/breast-cancer-cell-segmentation"

NREPEATS = 5   # repetições da CV (protocolo 5x10-fold). O exemplo do artigo usa 10.
QUICK = True   # True = teste rápido (todos os datasets, mínimo); False = run completo

In [ ]:
# --- Clonar + instalar (com o extra hopfield) ---
import os
os.environ["PYTHONUTF8"] = "1"
# Sair da pasta antes de apagá-la: num 2o Run All o kernel pode estar DENTRO de
# /kaggle/working/hopmil, e apagar o cwd quebra o getcwd (clone/pip falham).
%cd /kaggle/working
!rm -rf /kaggle/working/hopmil
!git clone -q --branch $BRANCH $REPO_URL /kaggle/working/hopmil
%cd /kaggle/working/hopmil
!pip install -q -e ".[hopfield]"
print("commit:")
!git rev-parse --short HEAD

In [ ]:
# --- W&B via Kaggle Secret ---
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
print("WANDB_API_KEY carregado:", bool(os.environ.get("WANDB_API_KEY")))

In [ ]:
# --- Rodar os experimentos de imagem (1 job por GPU, em paralelo) ---
import os
import subprocess
import time
from collections import deque

import torch

project = "artigo-3-smoke" if QUICK else "artigo-3"

common = [
    "python", "-m", "hopmil.eval.compare",
    "trainer.accelerator=gpu", "trainer.devices=1",
    "n_jobs=1",                      # 1 GPU por processo -> folds sequenciais nele
    "wandb.mode=online", f"wandb.project={project}",
]
if QUICK:
    common += ["cv.n_repeats=1", "cv.n_folds=2", "trainer.max_epochs=2", "early_stopping.patience=2"]
else:
    common += [f"cv.n_repeats={NREPEATS}", "cv.n_folds=10"]   # protocolo 5x10-fold

jobs = [
    {"data": "mnist_bags"},
    {"data": "fashion_mnist_bags"},
    {"data": "cifar_bags"},
    {"data": "colon_cancer", "root": COLON_ROOT},
    {"data": "ucsb_breast", "root": UCSB_ROOT},
]

# Lanes = uma por GPU física (T4 x2 -> mnist numa, cifar noutra, em paralelo).
n_gpu = torch.cuda.device_count()
gpus = list(range(n_gpu)) if n_gpu > 0 else [0]
print(f"GPUs detectadas: {n_gpu} -> {len(gpus)} lane(s) paralela(s): {gpus}", flush=True)

# Pula real-histo se o root não existir (não quebra).
queue = deque()
for j in jobs:
    if "root" in j and not os.path.isdir(j["root"]):
        print(f"== PULANDO {j['data']}: root não encontrado ({j['root']}) ==", flush=True)
        continue
    queue.append(j)


def _launch(gpu, j):
    args = list(common) + [f"data={j['data']}"]
    if "root" in j:
        args.append(f"data.root={j['root']}")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))  # processo enxerga só ESTA GPU
    log_path = f"/kaggle/working/log_{j['data']}.txt"
    fh = open(log_path, "w")
    print(f"[GPU {gpu}] >>> iniciando {j['data']}", flush=True)
    proc = subprocess.Popen(args, env=env, stdout=fh, stderr=subprocess.STDOUT)
    return {"proc": proc, "job": j, "log": log_path, "fh": fh}


running = {}
for gpu in gpus:
    if queue:
        running[gpu] = _launch(gpu, queue.popleft())

while running:
    time.sleep(20)
    for gpu in list(running):
        st = running[gpu]
        rc = st["proc"].poll()
        if rc is None:  # ainda rodando -> heartbeat com a última linha do log
            try:
                tail = open(st["log"]).read().strip().splitlines()[-1][-120:]
            except Exception:
                tail = "(sem saída ainda)"
            print(f"[GPU {gpu}] {st['job']['data']}: {tail}", flush=True)
        else:
            st["fh"].close()
            print(f"[GPU {gpu}] CONCLUÍDO {st['job']['data']} (exit {rc})", flush=True)
            if queue:
                running[gpu] = _launch(gpu, queue.popleft())
            else:
                del running[gpu]

print("== Todos os jobs terminaram. ==", flush=True)

In [ ]:
# --- Coletar os CSVs em /kaggle/working/results (vira output baixável) ---
os.makedirs("/kaggle/working/results", exist_ok=True)
for f in glob.glob("/kaggle/working/hopmil/results/*.csv"):
    shutil.copy(f, "/kaggle/working/results/")
print("CSVs salvos:")
!ls -la /kaggle/working/results

## Baixar os resultados
- **W&B**: runs no projeto escolhido (um por dataset, nome = dataset).
- **CSVs**: aba *Output* do notebook → `results/` (ou *Save Version* para persistir
  como output da versão). Cada dataset gera `*_folds.csv`, `*_summary.csv`,
  `*_bayesian.csv` com a suíte completa de métricas.